# Notebook 01: Exploratory Data Analysis (EDA)
## Indian Consumer Complaint Classification (INGRAM & RBI Ombudsman Domain)

This notebook explores the dataset distribution, missing records, complaint text lengths, vocabulary frequencies, and n-gram patterns across 5 core Indian consumer grievance sectors.

In [1]:
import os
import sys
import csv
from collections import Counter, defaultdict

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from src.preprocess import clean_text, tokenize
from src.features import get_ngrams

data_path = os.path.abspath('../data/processed/cleaned_complaints.csv')
records = []
with open(data_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for r in reader:
        records.append(r)

print(f'Total Cleaned Records: {len(records)}')

In [2]:
# 1. Category Distribution
category_counts = Counter(r['category'] for r in records)
print('Class Distribution:')
for cat, count in category_counts.most_common():
    pct = (count / len(records)) * 100
    print(f'  {cat:32s}: {count:4d} ({pct:5.1f}%)')

In [3]:
# 2. Text Length Statistics (Word Token Counts)
lengths = [int(r['token_count']) for r in records]
min_len = min(lengths)
max_len = max(lengths)
avg_len = sum(lengths) / len(lengths)
sorted_lens = sorted(lengths)
median_len = sorted_lens[len(sorted_lens) // 2]

print(f'Token Count Stats: Min={min_len}, Max={max_len}, Mean={avg_len:.1f}, Median={median_len}')

In [4]:
# 3. Top Salient N-Grams per Category
category_tokens = defaultdict(list)
for r in records:
    words = tokenize(r['cleaned_text'])
    ngrams = words + get_ngrams(words, 2)
    category_tokens[r['category']].extend(ngrams)

print('Top Discriminative Tokens per Category:')
for cat, tokens in category_tokens.items():
    top5 = Counter(tokens).most_common(5)
    tokens_str = ', '.join([f'{w} ({c})' for w, c in top5])
    print(f'\n[{cat}]:\n  {tokens_str}')